# 5th Notebook. To be run after feature extraction
## Computes KL divergence of feature means across trial and inter-trial inverals from full-length mask feature vectors.

## NEED TO ADD:
- TC/PC processing differentiation. TC can use current. PC needs to be using PC trial definitions of light. Will need to use exp log to get a give video's day's trial definitions (steal logic from NB2).

## STEP 1: Define KL functions.
This block defines two mathematical approaches to computing KL divergence:

1. **kl_gaussian_diag()** - "Diagonal" or "Independent Features" approach
    This assumes that each feature varies independently (probably not true) and treats the 10-dimensional
    distribution as 10 separate 1D Gaussians. Use when features don't strongly correlate.
    Formula: KL = Σ[0.5 * (log(σ²_B/σ²_A) + (σ²_A + (μ_A - μ_B)²)/σ²_B - 1)]



2. **kl_gaussian_full()** - "Full Covariance" or "Multivariate" approach
    This version accounts for correlations between features; the 10-dimensional distribution is treated as one 
    multivariate Gaussian, which requires computing and inverting the 10x10 covariance matrix (yikes!). This 
    approach is data hungry, but should be used when features have known correlations, which is certainly the 
    case here.
    Formula: KL = 0.5 * [tr(Σ_B⁻¹Σ_A) + (μ_B-μ_A)ᵀΣ_B⁻¹(μ_B-μ_A) - d + log(det(Σ_B)/det(Σ_A))]


**Why both?**
- Diagonal is robust but may underestimate divergence if correlations differ between conditions
- Full is theoretically correct but can be numerically unstable with ~2000 samples in 10 dimensions
- Comparing both helps validate results

**Returns:**
- Total KL divergence (scalar)
- Per-feature contributions (array) - shows which features drive the divergence

# Imports

In [9]:
import numpy as np
import pandas as pd
import os
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pdb
import glob
import datetime
from collections import defaultdict
import traceback

# Define session to process

In [10]:
VIDEO_PREFIX_FILTER = ['2025_10_14_10_25_19']
# VIDEO_PREFIX_FILTER = ['2025_10_17_14_30' ,
#                        '2025_10_17_10_18_17',
#                        '2025_10_16_14_28_16',
#                        '2025_10_16_10_10_37',
#                        '2025_10_15_14_30_16',
#                        '2025_10_15_10_20_58',
#                        '2025_10_14_14_25_09',
#                        '2025_10_14_10_35_03'] # Feature files starting with this will be processed
VIDEO_GROUP_FILTER = 'TC'              # Session group: 'TC', 'TP', or None for all groups
EXCLUSION_FILTER = []                  # Exclude any files/sessions containing these substrings
                                       # Uses SUBSTRING matching
                                       # Example: 'badvideo' excludes files with 'badvideo' anywhere in name
LABEL_TYPE = 'points'

# Define functions

In [11]:
def kl_gaussian_diag(A: np.ndarray, B: np.ndarray, eps: float = 1e-12) -> float:
    """
    Compute KL divergence KL(A || B) assuming diagonal (independent) Gaussians.
    
    Parameters:
    -----------
    A : np.ndarray, shape (n_samples, n_features)
        Samples from distribution A
    B : np.ndarray, shape (m_samples, n_features)
        Samples from distribution B
    eps : float
        Small value added to variances for numerical stability
    
    Returns:
    --------
    float: KL(A || B) = sum of 1D Gaussian KL divergences
    array: Per-feature KL contributions
    
    Formula for each feature:
    KL = 0.5 * [ log(var_B/var_A) + (var_A + (mean_A - mean_B)^2)/var_B - 1 ]
    """
    # Per-feature means
    mu_A = np.nanmean(A, axis=0)  # Shape: (n_features,)
    mu_B = np.nanmean(B, axis=0)
    
    # Per-feature variances (unbiased estimator with ddof=1)
    var_A = np.nanvar(A, axis=0, ddof=1) + eps    # eps = 1e-12, used to prevent div by zero
    var_B = np.nanvar(B, axis=0, ddof=1) + eps
    
    # Sum of 1D KL divergences
    kl_per_feature = 0.5 * (np.log(var_B / var_A) + (var_A + (mu_A - mu_B)**2) / var_B - 1.0)
    kl_total = np.sum(kl_per_feature)
    
    return kl_total, kl_per_feature


def kl_gaussian_full(A: np.ndarray, B: np.ndarray, eps: float = 1e-6) -> float:
    """
    Compute KL divergence KL(A || B) for full multivariate Gaussians.
    
    Parameters:
    -----------
    A : np.ndarray, shape (n_samples, n_features)
        Samples from distribution A
    B : np.ndarray, shape (m_samples, n_features)
        Samples from distribution B
    eps : float
        Small value added to diagonal of covariance for stability
    
    Returns:
    --------
    float: KL(A || B)
    
    Formula:
    KL = 0.5 * [ tr(Sigma_B^{-1} Sigma_A) + (mu_B - mu_A)^T Sigma_B^{-1} (mu_B - mu_A) 
                 - d + log(det(Sigma_B) / det(Sigma_A)) ]
    """
    n_samples, d = A.shape
    
    # Estimate means
    mu_A = np.nanmean(A, axis=0)
    mu_B = np.nanmean(B, axis=0)
    
    # Estimate covariance matrices (add eps*I for numerical stability)
    Sigma_A = np.cov(A, rowvar=False) + eps * np.eye(d)
    Sigma_B = np.cov(B, rowvar=False) + eps * np.eye(d)
    
    # Compute KL divergence terms
    inv_Sigma_B = np.linalg.inv(Sigma_B)
    diff = mu_B - mu_A
    
    trace_term = np.trace(inv_Sigma_B @ Sigma_A)
    mahalanobis_term = diff.T @ inv_Sigma_B @ diff
    logdet_term = np.log(np.linalg.det(Sigma_B) / np.linalg.det(Sigma_A))
    
    kl_div = 0.5 * (trace_term + mahalanobis_term - d + logdet_term)
    
    return kl_div


print("KL divergence functions loaded.")

KL divergence functions loaded.


## Extract the on and off frame indexes

In [12]:
def extract_CSon_CSoff_frames(Feature_vector, trial_definitions_df, feature_names, verbose=True):
    """
    Extract CS-on and CS-off frames from full feature vector.
    
    Parameters:
    -----------
    Feature_vector : np.ndarray, shape (10, n_frames)
        Full video feature array
    trial_definitions_df : pd.DataFrame
        Trial timing information
    feature_names : list
        Names of the 10 features
    verbose : bool
        If True, prints detailed extraction information. If False, runs silently.
    
    Returns:
    --------
    CS_on_samples : np.ndarray, shape (n_CS_on_frames, 10)
        CS-on data in (samples, features) format
    CS_off_samples : np.ndarray, shape (n_CS_off_frames, 10)
        CS-off data in (samples, features) format
    extraction_info : dict
        Metadata about extraction process
    """
    
    if verbose:
        print("="*30)
        print("Extracting CS-on frames (during trials)...")
    
    CS_on_frames = []
    CS_on_frame_indices = []
    
    for idx, row in trial_definitions_df.iterrows():
        trial_num = row['trial_num']
        start_frame = row['start_frame']
        end_frame = row['end_frame']
        
        # Extract frames for this trial
        trial_frames = Feature_vector[:, start_frame:end_frame+1]  # Shape: (10, num_frames_in_trial)
        CS_on_frames.append(trial_frames)
        CS_on_frame_indices.extend(range(start_frame, end_frame+1))
        
        if verbose:
            print(f"  Trial {trial_num}: frames {start_frame}-{end_frame} ({end_frame - start_frame + 1} frames)")
    
    # Concatenate all CS-on frames
    CS_on_data = np.concatenate(CS_on_frames, axis=1)  # Shape: (10, total_CS_on_frames)
    if verbose:
        print(f"Total CS-on frames: {CS_on_data.shape[1]}")
    
    # Extract CS-off frames
    if verbose:
        print("\nExtracting CS-off frames (inter-trial intervals only)...")
    
    CS_off_frames = []
    CS_off_frame_indices = []

    # If doesn't include the last off-ITI, include it!
    off_sess_num = len(trial_definitions_df) - 1
    if int(trial_definitions_df['end_frame'][len(trial_definitions_df)-1]) != len(Feature_vector[0]):
        off_sess_num = off_sess_num + 1

    # Inter-trial intervals (between consecutive trials)
    for idx in range(off_sess_num):
        if idx == (off_sess_num-1):
            interval_start = trial_definitions_df.iloc[idx]['end_frame']+1
            interval_end = len(Feature_vector[0])
        else:
            current_trial = trial_definitions_df.iloc[idx]
            next_trial = trial_definitions_df.iloc[idx + 1]
            
            interval_start = current_trial['end_frame'] + 1
            interval_end = next_trial['start_frame'] - 1
        
        if interval_end >= interval_start:
            interval_frames = Feature_vector[:, interval_start:interval_end+1]
            CS_off_frames.append(interval_frames)
            CS_off_frame_indices.extend(range(interval_start, interval_end+1))
            
            if verbose:
                #print(f"  After trial {current_trial['trial_num']}: frames {interval_start}-{interval_end} ({interval_end - interval_start + 1} frames)")
                print(f"  After trial {str(idx+1)}: frames {interval_start}-{interval_end} ({interval_end - interval_start + 1} frames)")

        else:
            if verbose:
                print(f"  After trial {str(idx+1)}: No interval (trials adjacent)") # See alt if doesnt work
    
    # Concatenate all CS-off frames
    if CS_off_frames:
        CS_off_data = np.concatenate(CS_off_frames, axis=1)  # Shape: (10, total_CS_off_frames)
        if verbose:
            print(f"Total CS-off frames: {CS_off_data.shape[1]}")
    else:
        raise ValueError("No CS-off frames found! Trials may be adjacent with no inter-trial intervals.")
    
    # Transpose to (samples, features) format
    if verbose:
        print("\nTransposing to (samples, features) format...")
    CS_on_samples = CS_on_data.T   # Shape: (n_CS_on_frames, 10)
    CS_off_samples = CS_off_data.T  # Shape: (n_CS_off_frames, 10)
    
    if verbose:
        print(f"  CS-on:  {CS_on_samples.shape}")
        print(f"  CS-off: {CS_off_samples.shape}")
    
    # NaN analysis
    if verbose:
        print("\nNaN analysis:")
    CS_on_nans = np.sum(np.isnan(CS_on_samples), axis=0)
    CS_off_nans = np.sum(np.isnan(CS_off_samples), axis=0)
    
    if verbose:
        print(f"{'Feature':<20} {'CS-on NaNs':<15} {'CS-off NaNs':<15} {'CS-on %':<10} {'CS-off %':<10}")
        print("-"*70)
        for i, fname in enumerate(feature_names):
            cs_on_pct = 100 * CS_on_nans[i] / CS_on_samples.shape[0]
            cs_off_pct = 100 * CS_off_nans[i] / CS_off_samples.shape[0]
            print(f"{fname:<20} {CS_on_nans[i]:<15} {CS_off_nans[i]:<15} {cs_on_pct:<10.2f} {cs_off_pct:<10.2f}")
    
    # Store extraction info
    extraction_info = {
        'n_CS_on_frames': CS_on_samples.shape[0],
        'n_CS_off_frames': CS_off_samples.shape[0],
        'CS_on_frame_indices': CS_on_frame_indices,
        'CS_off_frame_indices': CS_off_frame_indices,
        'CS_on_nans_per_feature': CS_on_nans.tolist(),
        'CS_off_nans_per_feature': CS_off_nans.tolist()
    }
    
    return CS_on_samples, CS_off_samples, extraction_info

print("Data extraction functions defined.")

Data extraction functions defined.


## STEP 3: Define more KL functions.

This block contains two functions that compute and format KL divergence results:

**compute_kl_divergence():**
Calculates four different KL divergence values:

1. **KL(CS-on || CS-off)** - PRIMARY METRIC
   - "How much information is lost if we approximate CS-on with CS-off?"
   - Interpretation: How different is the worm's behavior during stimulus vs. rest?
   - Higher = more divergent (worm changes more during stimulus)
   - This is asymmetric: measures divergence FROM CS-on's perspective

2. **KL(CS-off || CS-on)** - REVERSE METRIC
   - Same question but from CS-off's perspective
   - Usually different from KL(CS-on || CS-off) due to asymmetry
   - Higher = CS-off is further from CS-on

3. **KL_symmetric** - SYMMETRIC VERSION
   - Average of both directions: 0.5 * [KL(A||B) + KL(B||A)]
   - Symmetric (doesn't depend on order)
   - Use when you want a single "distance" metric
   - Higher = distributions are more different (regardless of direction)

4. **KL_full_covariance** - WITH CORRELATIONS
   - Uses kl_gaussian_full() instead of kl_gaussian_diag()
   - Accounts for feature correlations (e.g., area and perimeter co-vary)
   - May fail if covariance matrix is singular (insufficient data or perfect collinearity)
   - If successful, should be ≥ diagonal version (correlations add information)

**Per-feature breakdown:**
- Shows which features contribute most to total KL
- Example: If "Areas" contributes 60% of KL, area changes drive the divergence
- Helps interpret WHY distributions differ

**create_results_dataframe():**
Formats all results into a pandas DataFrame for easy inspection and saving:
- One row per video
- Columns: video name, session, frame counts, all 4 KL values, per-feature contributions
- Easy to concatenate results from multiple videos for batch analysis

**Returns:**
- results_dict: Python dictionary with all KL values and metadata
- results_df: Pandas DataFrame ready for CSV export or further analysis

In [13]:
def compute_kl_divergence(CS_on_samples, CS_off_samples, feature_names):
    """a
    Compute KL divergence between CS-on and CS-off distributions.
    
    Parameters:
    -----------
    CS_on_samples : np.ndarray, shape (n_samples, n_features)
        CS-on data
    CS_off_samples : np.ndarray, shape (m_samples, n_features)
        CS-off data
    feature_names : list
        Names of features
    
    Returns:
    --------
    results_dict : dict
        All KL divergence results and metadata
    """
    
    print("="*30)
    print("Computing KL divergence...")
    
    # Primary: KL(CS-on || CS-off)
    kl, kl_per_feature = kl_gaussian_diag(CS_on_samples, CS_off_samples)
    print(f"\nKL(CS-on || CS-off) = {kl:.6f}")

    
    # Reverse: KL(CS-off || CS-on)
    #kl_off_given_on, kl_per_feature_reverse = kl_gaussian_diag(CS_off_samples, CS_on_samples)
    #print(f"KL(CS-off || CS-on) = {kl_off_given_on:.6f}")
    
    # Symmetric KL
    #kl_symmetric = 0.5 * (kl_on_given_off + kl_off_given_on)
    #print(f"KL_symmetric        = {kl_symmetric:.6f}")
    
    # Full covariance version (optional)
    #try:
    #    kl_full = kl_gaussian_full(CS_on_samples, CS_off_samples)
    #    print(f"KL_full (with correlations) = {kl_full:.6f}")
    #except np.linalg.LinAlgError:
    #    print(f"KL_full computation failed (singular covariance matrix)")
    #    kl_full = np.nan
    
    # Per-feature contributions
    print(f"\n{'='*30}")
    print(f"Per-feature KL contributions:")
    print(f"{'='*30}")
    print(f"{'Feature':<20} {'KL contrib':<15} {'% of total':<12}")
    print("-"*50)
    
    #per_feature_results = {}
    #for i, fname in enumerate(feature_names):
    #    pct = 100 * kl_per_feature[i] / kl_on_given_off if kl_on_given_off != 0 else 0
    #    print(f"{fname:<20} {kl_per_feature[i]:<15.6f} {pct:<12.2f}")
    #    per_feature_results[fname] = {
    #        'kl_contribution': float(kl_per_feature[i]),
    #        'percent_of_total': float(pct)
    #    }
    
    # Compile results
    results_dict = {
        'kl_all': float(kl),
        'kl_per_feature' : kl_per_feature
        #'kl_off_given_on': float(kl_off_given_on),
        #'kl_symmetric': float(kl_symmetric),
        #'kl_full_covariance': float(kl_full) if not np.isnan(kl_full) else None,
        #'per_feature_contributions': per_feature_results
    }
    
    return results_dict


print("KL computation and results functions defined.")

KL computation and results functions defined.


## STEP 4: Define functions to create null dsitribution with circshift.

**Purpose:** Implement exhaustive circular shift permutation testing to assess statistical significance of KL divergence values.

**What it does:**
This block provides three functions for null hypothesis testing:

1. **compute_circshift_null_distribution()** - Generates complete null distribution
   - Computes all possible circular shifts (1 to num_frames-1, typically ~2039 shifts for 2040-frame videos)
   - For each shift: circularly shifts all 10 features synchronously (preserving cross-feature correlations), extracts CS-on/CS-off frames using original trial timing applied to shifted data, computes KL divergence for all three measures
   - Returns arrays of null KL values for: KL(CS-on||CS-off), KL(CS-off||CS-on), KL_symmetric

2. **plot_circshift_diagnostic()** - Visualizes temporal alignment effect
   - Creates 3-panel plot of KL vs shift amount
   - If temporal alignment matters: expect peak at shift=0/2040, trough near shift=1020
   - If no relationship exists: expect flat line across all shifts
   - Helps visually confirm whether observed KL is driven by true temporal structure

3. **compute_null_statistics()** - Calculates p-values and summary stats
   - **One-tailed p-value**: Proportion of null KL values ≥ observed (tests if observed is significantly higher than chance)
   - **Two-tailed p-value**: 2 × min(proportion ≤ observed, proportion ≥ observed) (tests if observed differs in either direction)
   - Returns null mean, std, median, min, max, and both p-values

**Output:**
- Null statistics added to CSV: `null_mean_*`, `null_std_*`, `p_one_tailed_*`, `p_two_tailed_*` for each KL measure
- Optional: Commented-out code to save complete null distributions as .npz files (~2039 values per video)

**Computational note:**
- Exhaustive testing is feasible because num_shifts ≈ 2040 (takes ~30-60s per video)
- This is preferable to random sampling because it gives exact p-values without Monte Carlo error


- Although I'm computing those indivudal p-values here for each video, they're not the actual statistical test we'll be using. The real test will come from the distribution of KL values we get from all the videos together. 

In [14]:
def compute_circshift_null_distribution(Feature_vector, 
                                       trial_definitions_df, 
                                       feature_names,
                                       show_diagnostic_plot=True):
    """
    Compute complete circshift null distribution for KL divergence.
    
    For each possible circular shift (1 to num_frames-1), this function:
    1. Circularly shifts the entire feature matrix along the time axis
    2. Extracts CS-on and CS-off frames using the ORIGINAL (unshifted) trial timing
    3. Computes KL divergence between the shifted CS-on and CS-off distributions
    
    This creates a null distribution representing KL values you'd expect if there
    were no true temporal relationship between morphology and trial timing.
    
    Parameters:
    -----------
    Feature_vector : np.ndarray, shape (10, n_frames)
        Full video feature array
    trial_definitions_df : pd.DataFrame
        Trial timing information (used unshifted for all iterations)
    feature_names : list
        Names of the 10 features
    show_diagnostic_plot : bool
        Whether to display KL vs shift diagnostic plot
    
    Returns:
    --------
    null_results : dict
        Contains:
        - 'null_kl_on_given_off': array of KL(CS-on || CS-off) for all shifts
        - 'null_kl_off_given_on': array of KL(CS-off || CS-on) for all shifts
        - 'null_kl_symmetric': array of symmetric KL for all shifts
        - 'shift_values': array of shift amounts used
    """
    
    print("\n" + "="*70)
    print("COMPUTING CIRCSHIFT NULL DISTRIBUTION")
    print("="*70)
    
    num_frames = Feature_vector.shape[1]
    num_shifts = num_frames - 1  # All shifts from 1 to num_frames-1
    
    print(f"Total frames: {num_frames}")
    print(f"Number of circshifts to compute: {num_shifts}")
    print("Computing null distribution with live updates...")
    print()
    
    # Initialize arrays to store null KL values
    null_kl_on_given_off = np.zeros(num_shifts)
    null_kl_off_given_on = np.zeros(num_shifts)
    null_kl_symmetric = np.zeros(num_shifts)
    shift_values = np.arange(1, num_frames)
    
    # Compute KL for each circshift with live console updates
    for i, shift in enumerate(shift_values):
        # Circularly shift the feature matrix along time axis (axis=1)
        # All 10 features shift together by the same amount
        shifted_features = np.roll(Feature_vector, shift=shift, axis=1)
        
        # Extract CS-on and CS-off using ORIGINAL trial timing on SHIFTED data
        # verbose=False suppresses the detailed output
        CS_on_samples_shifted, CS_off_samples_shifted, _ = extract_CSon_CSoff_frames(
            shifted_features, 
            trial_definitions_df, 
            feature_names,
            verbose=False
        )
        
        # Compute KL divergence for this shift
        kl_on_off, _ = kl_gaussian_diag(CS_on_samples_shifted, CS_off_samples_shifted)
        kl_off_on, _ = kl_gaussian_diag(CS_off_samples_shifted, CS_on_samples_shifted)
        kl_sym = 0.5 * (kl_on_off + kl_off_on)
        
        null_kl_on_given_off[i] = kl_on_off
        null_kl_off_given_on[i] = kl_off_on
        null_kl_symmetric[i] = kl_sym
        
        # Live update: compute running statistics and display on same line
        current_data = null_kl_on_given_off[:i+1]
        running_mean = np.mean(current_data)
        running_std = np.std(current_data)
        running_min = np.min(current_data)
        running_max = np.max(current_data)
        
        # Use \r to return to start of line and overwrite
        print(f"\r  Progress: {i+1:4d}/{num_shifts} | "
              f"Mean: {running_mean:.5f} | "
              f"Std: {running_std:.5f} | "
              f"Min: {running_min:.5f} | "
              f"Max: {running_max:.5f}", 
              end='', flush=True)
    
    # Print newline after loop completes to move to next line
    print()
    
    print(f"\nCircshift null distribution computed successfully!")
    print(f"\nFinal statistics for KL(CS-on || CS-off):")
    print(f"  Mean:   {np.mean(null_kl_on_given_off):.6f}")
    print(f"  Std:    {np.std(null_kl_on_given_off):.6f}")
    print(f"  Median: {np.median(null_kl_on_given_off):.6f}")
    print(f"  Range:  [{np.min(null_kl_on_given_off):.6f}, {np.max(null_kl_on_given_off):.6f}]")
    
    # Optionally plot diagnostic
    if show_diagnostic_plot:
        plot_circshift_diagnostic(shift_values, null_kl_on_given_off, 
                                 null_kl_off_given_on, null_kl_symmetric)
    
    null_results = {
        'null_kl_on_given_off': null_kl_on_given_off,
        'null_kl_off_given_on': null_kl_off_given_on,
        'null_kl_symmetric': null_kl_symmetric,
        'shift_values': shift_values
    }
    
    return null_results


def plot_circshift_diagnostic(shift_values, null_kl_on_given_off, 
                              null_kl_off_given_on, null_kl_symmetric):
    """
    Plot KL divergence vs. shift amount to visualize temporal alignment effect.
    
    If there's a true relationship between morphology and trial timing:
    - Peak KL should occur at shift=0 (true alignment)
    - Minimum KL should occur around shift=num_frames/2 (maximal misalignment)
    
    If there's no relationship:
    - KL should be roughly flat across all shift values
    
    Parameters:
    -----------
    shift_values : np.ndarray
        Array of shift amounts
    null_kl_on_given_off : np.ndarray
        KL(CS-on || CS-off) for each shift
    null_kl_off_given_on : np.ndarray
        KL(CS-off || CS-on) for each shift
    null_kl_symmetric : np.ndarray
        Symmetric KL for each shift
    """
    
    fig, axes = plt.subplots(3, 1, figsize=(12, 10))
    
    # Plot each KL measure
    axes[0].plot(shift_values, null_kl_on_given_off, linewidth=0.8, color='blue', alpha=0.7)
    axes[0].axhline(np.mean(null_kl_on_given_off), color='red', linestyle='--', 
                    linewidth=1, label=f'Mean = {np.mean(null_kl_on_given_off):.4f}')
    axes[0].set_ylabel('KL(CS-on || CS-off)', fontsize=11)
    axes[0].set_title('Circshift Null Distribution: KL vs Shift Amount', fontsize=12, fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(shift_values, null_kl_off_given_on, linewidth=0.8, color='green', alpha=0.7)
    axes[1].axhline(np.mean(null_kl_off_given_on), color='red', linestyle='--', 
                    linewidth=1, label=f'Mean = {np.mean(null_kl_off_given_on):.4f}')
    axes[1].set_ylabel('KL(CS-off || CS-on)', fontsize=11)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    axes[2].plot(shift_values, null_kl_symmetric, linewidth=0.8, color='purple', alpha=0.7)
    axes[2].axhline(np.mean(null_kl_symmetric), color='red', linestyle='--', 
                    linewidth=1, label=f'Mean = {np.mean(null_kl_symmetric):.4f}')
    axes[2].set_xlabel('Shift Amount (frames)', fontsize=11)
    axes[2].set_ylabel('KL_symmetric', fontsize=11)
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\nDiagnostic plot explanation:")
    print("  - If temporal alignment matters: expect peak near shift=0, trough near middle")
    print("  - If no temporal relationship: expect roughly flat line across all shifts")


def compute_null_statistics(observed_kl, null_kl_distribution):
    """
    Compute p-values and summary statistics for observed KL vs null distribution.
    
    P-value calculation:
    -------------------
    One-tailed (upper): What proportion of null KL values are >= observed KL?
        - Tests: "Is the observed KL significantly HIGHER than expected by chance?"
        - Formula: p = (# null values >= observed) / (total # null values)
        - Small p-value means observed KL is unusually high (suggests real effect)
    
    Two-tailed: What proportion of null KL values are as extreme or more extreme?
        - Tests: "Is the observed KL significantly DIFFERENT from null (in either direction)?"
        - Formula: p = 2 * min(p_lower, p_upper), capped at 1.0
        - Where p_lower = proportion of null values <= observed
        - And p_upper = proportion of null values >= observed
        - This is a two-sided test treating both tails as evidence against null hypothesis
    
    Parameters:
    -----------
    observed_kl : float
        The KL divergence computed from true (unshifted) temporal alignment
    null_kl_distribution : np.ndarray
        Array of KL values from all circshifts
    
    Returns:
    --------
    stats : dict
        Contains p-values and null distribution summary statistics
    """
    
    n_null = len(null_kl_distribution)
    
    # One-tailed p-value (upper tail)
    # How many null values are >= observed?
    n_greater_equal = np.sum(null_kl_distribution >= observed_kl)
    p_value_one_tailed = n_greater_equal / n_null
    
    # Two-tailed p-value
    # How many null values are <= observed (lower tail)?
    n_less_equal = np.sum(null_kl_distribution <= observed_kl)
    p_lower = n_less_equal / n_null
    p_upper = n_greater_equal / n_null
    p_value_two_tailed = 2 * min(p_lower, p_upper)
    p_value_two_tailed = min(p_value_two_tailed, 1.0)  # Cap at 1.0
    
    stats = {
        'p_value_one_tailed': p_value_one_tailed,
        'p_value_two_tailed': p_value_two_tailed,
        'null_mean': np.mean(null_kl_distribution),
        'null_std': np.std(null_kl_distribution),
        'null_median': np.median(null_kl_distribution),
        'null_min': np.min(null_kl_distribution),
        'null_max': np.max(null_kl_distribution),
        'observed_kl': observed_kl
    }
    
    return stats


print("Circshift null distribution functions loaded.")

Circshift null distribution functions loaded.


## STEP 5: Define master use function.

**analyze_video_kl_divergence()** - The top-level coordinator

**Purpose:**
Orchestrates the entire KL divergence pipeline for a single video. This function:
1. Validates inputs (checks files exist)
2. Loads data (feature vector + trial definitions)
3. Calls extraction function (Block 2)
4. Calls KL computation function (Block 3)
5. **Optionally computes circshift null distribution (Block 3.5)**
6. **Calculates p-values comparing observed KL to null distribution**
7. Saves results to JSON and CSV
8. Returns formatted results

**New Parameters:**
- `compute_null` (bool): Whether to compute exhaustive circshift null distribution (~30-60s per video)
- `show_null_diagnostic` (bool): Whether to display KL vs shift diagnostic plot

**Output additions:**
- If `compute_null=True`, adds null statistics columns to CSV: null mean/std, one-tailed p-value, two-tailed p-value (for all three KL measures)
- Diagnostic plot shows whether temporal alignment matters (peak at shift=0, trough at shift~1020) or doesn't matter (flat line)

In [ ]:
def _parse_start_end_from_row(row):
    # Rejoin everything after the first column
    rest = ",".join(row[1:]).strip()
    # Find all [...] groups
    groups = re.findall(r"\[(.*?)\]", rest)
    starts, ends = [], []
    if len(groups) >= 1:
        starts = [int(x) for x in groups[0].split(",") if x.strip()]
    if len(groups) >= 2:
        ends   = [int(x) for x in groups[1].split(",") if x.strip()]
    return starts, ends

def get_session_splits(csv_path, session_name):
    with open(csv_path, newline="") as f:
        reader = csv.reader(f)
        for row in reader:
            if not row:
                continue
            date = row[0].strip()
            if date != session_name:
                continue
            return _parse_start_end_from_row(row)
    raise ValueError(f"Session {session_name!r} not found in CSV")

# --- Usage example ---

start_frames, end_frames = get_session_splits(csv_path, session)

def remove_poor_values(feature_vector,session):
    csv_path = "/n/holylabs/gershman_lab/Users/zkelso/video_splits.csv"
    start_frames, end_frames = get_session_splits(csv_path, session)
    
    pdb.set_trace()
    for s, e in zip(start_frames, end_frames):
        feature_vector[:, s:e+1] = np.nan

    return feature_vector


In [15]:
def analyze_video_kl_divergence(video_name, 
                                home_dir, 
                                netscratch_dir,LABEL_TYPE,
                                save_results=True,
                                output_dir=None,
                                compute_null=False,
                                show_null_diagnostic=True):
    """
    Master function to compute KL divergence between CS-on and CS-off for a single video.
    
    Parameters:
    -----------
    video_name : str
        Video name (without _Feature_vector.npy extension)
    home_dir : str
        Path to HOME directory
    netscratch_dir : str
        Path to NETSCRATCH directory
    save_results : bool
        Whether to save results to JSON file
    output_dir : str, optional
        Where to save results (default: NETSCRATCH/KL_Results)
    compute_null : bool
        Whether to compute circshift null distribution (adds ~30-60s per video)
    show_null_diagnostic : bool
        Whether to display diagnostic plot of KL vs shift (only if compute_null=True)
    
    Returns:
    --------
    results_df : pd.DataFrame
        Summary results
    full_results : dict
        Complete results including all metadata
    """
    
    print("="*70)
    print("KL DIVERGENCE ANALYSIS")
    print("="*70)
    print(f"Video: {video_name}")
    
    # Feature names
    feature_names = ['Areas', 'Area_percentages', 'Perimeters', 'Area_perimeter_ratios',
                    'Circularities', 'Hull_areas', 'Centroidxs', 'Centroidys', 
                    'Angles', 'Concavities','PC1','PC2']
    
    # Construct paths
    features_folder = os.path.join(netscratch_dir, 'Features')
    raw_data_folder = os.path.join(home_dir, 'Raw_data')

    feature_file = os.path.join(features_folder, f"{video_name}{LABEL_TYPE}_Feature_vector.npy")
    
    # Extract session prefix
    if "_regions_" in video_name:
        session_prefix = video_name.split("_regions_")[0]
    else:
        raise ValueError(f"Could not extract session prefix from: {video_name}")
    
    trial_csv_path = os.path.join(raw_data_folder, session_prefix, "trial_definitions.csv")
    
    print(f"Session: {session_prefix}")
    
    # Check files exist
    if not os.path.exists(feature_file):
        raise FileNotFoundError(f"Feature file not found: {feature_file}")
    
    if not os.path.exists(trial_csv_path):
        raise FileNotFoundError(f"Trial definitions not found: {trial_csv_path}")
    
    # Load data
    print("\nLoading data...")
    Feature_vector = np.load(feature_file)
    print(f"  Feature vector: {Feature_vector.shape}")
    
    trial_definitions_df = pd.read_csv(trial_csv_path)
    print(f"  Trial definitions: {len(trial_definitions_df)} trials")

    # Remove QC'd out portions of the feature vector
    pdb.set_trace()
    remove_poor_values(Feature_vector,video_name)
    
    # Extract CS-on and CS-off frames
    CS_on_samples, CS_off_samples, extraction_info = extract_CSon_CSoff_frames(
        Feature_vector, trial_definitions_df, feature_names
    )
    
    # Compute KL divergence
    results_dict = compute_kl_divergence(CS_on_samples, CS_off_samples, feature_names)
    
    # Compute null distribution if requested
    null_stats = None
    if compute_null:
        null_results = compute_circshift_null_distribution(
            Feature_vector, 
            trial_definitions_df, 
            feature_names,
            show_diagnostic_plot=show_null_diagnostic
        )
        
        # Compute statistics for each KL measure
        null_stats = {
            'on_given_off': compute_null_statistics(
                results_dict['kl_on_given_off'], 
                null_results['null_kl_on_given_off']
            ),
            'off_given_on': compute_null_statistics(
                results_dict['kl_off_given_on'], 
                null_results['null_kl_off_given_on']
            ),
            'symmetric': compute_null_statistics(
                results_dict['kl_symmetric'], 
                null_results['null_kl_symmetric']
            )
        }
        
        print("\n" + "="*70)
        print("NULL DISTRIBUTION STATISTICS")
        print("="*70)
        for kl_type, stats in null_stats.items():
            print(f"\n{kl_type.upper()}:")
            print(f"  Observed KL: {stats['observed_kl']:.6f}")
            print(f"  Null mean:   {stats['null_mean']:.6f}")
            print(f"  Null std:    {stats['null_std']:.6f}")
            print(f"  p-value (one-tailed): {stats['p_value_one_tailed']:.4f}")
            print(f"  p-value (two-tailed): {stats['p_value_two_tailed']:.4f}")
        
        # OPTIONAL: Save complete null distributions
        # Uncomment the following block to save all ~2039 null KL values per video
        # This creates large files but preserves complete null distributions
        """
        if save_results:
            if output_dir is None:
                output_dir = os.path.join(netscratch_dir, 'KL_Results')
            null_dist_file = os.path.join(output_dir, f"{video_name}_null_distributions.npz")
            np.savez_compressed(
                null_dist_file,
                shift_values=null_results['shift_values'],
                null_kl_on_given_off=null_results['null_kl_on_given_off'],
                null_kl_off_given_on=null_results['null_kl_off_given_on'],
                null_kl_symmetric=null_results['null_kl_symmetric']
            )
            print(f"\nNull distributions saved to: {null_dist_file}")
        """
    
    # Create results DataFrame
    results_df = create_results_dataframe(
        video_name, session_prefix, results_dict, feature_names, extraction_info, null_stats)
    
    # Compile full results
    full_results = {
        'video': video_name,
        'session': session_prefix,
        'extraction_info': extraction_info,
        'kl_results': results_dict,
        'null_statistics': null_stats
    }
    
    # Save results if requested
    if save_results:
        if output_dir is None:
            output_dir = os.path.join(netscratch_dir, 'KL_Results')
        
        os.makedirs(output_dir, exist_ok=True)
        
        # Save JSON
        json_path = os.path.join(output_dir, f"{video_name}_KL_results.json")
        with open(json_path, 'w') as f:
            json.dump(full_results, f, indent=2)
        
        print(f"\n{'='*70}")
        print(f"Results saved to: {json_path}")
        
        # Save CSV
        csv_path = os.path.join(output_dir, f"{video_name}_KL_results.csv")
        results_df.to_csv(csv_path, index=False)
        print(f"CSV saved to: {csv_path}")
    
    print(f"\n{'='*70}")
    print("Analysis complete")
    print("="*70)
    
    return results_df, full_results


def create_results_dataframe(video_name, session_prefix, results_dict, feature_names, extraction_info, null_stats=None):
    """
    Create a DataFrame with KL divergence results.
    
    Parameters:
    -----------
    video_name : str
        Name of the video analyzed
    session_prefix : str
        Session identifier
    results_dict : dict
        KL divergence results
    extraction_info : dict
        Frame extraction metadata
    null_stats : dict, optional
        Null distribution statistics (if compute_null=True)
    
    Returns:
    --------
    pd.DataFrame : Summary results
    """
    
    # Extract worm regions (coordinates) from video name
    try:
        if "_regions_" in video_name and "_fullvideo" in video_name:
            regions_part = video_name.split("_regions_")[1].split("_fullvideo")[0]
            worm_regions = regions_part
        else:
            worm_regions = "unknown"
    except Exception as e:
        print(f"Warning: Could not extract worm_regions from video name: {e}")
        worm_regions = "unknown"
    
    # Create main results row
    main_results = {
        'video': video_name,
        'session': session_prefix,
        'Troupe':'',
        'Day':'',
        'Block':'',
        'worm_regions': worm_regions,  
        'n_CSon_frames': extraction_info['n_CS_on_frames'],
        'n_CSoff_frames': extraction_info['n_CS_off_frames'],
        'KL': results_dict['kl_all'],
        'KL_per_feature': results_dict['kl_per_feature']
    }

    
    # Add null statistics if available
    if null_stats is not None:
        # For KL(CS-on || CS-off)
        main_results['null_mean_CSon_given_CSoff'] = null_stats['on_given_off']['null_mean']
        main_results['null_std_CSon_given_CSoff'] = null_stats['on_given_off']['null_std']
        main_results['p_one_tailed_CSon_given_CSoff'] = null_stats['on_given_off']['p_value_one_tailed']
        main_results['p_two_tailed_CSon_given_CSoff'] = null_stats['on_given_off']['p_value_two_tailed']
        
        # For KL(CS-off || CS-on)
        main_results['null_mean_CSoff_given_CSon'] = null_stats['off_given_on']['null_mean']
        main_results['null_std_CSoff_given_CSon'] = null_stats['off_given_on']['null_std']
        main_results['p_one_tailed_CSoff_given_CSon'] = null_stats['off_given_on']['p_value_one_tailed']
        main_results['p_two_tailed_CSoff_given_CSon'] = null_stats['off_given_on']['p_value_two_tailed']
        
        # For symmetric KL
        main_results['null_mean_symmetric'] = null_stats['symmetric']['null_mean']
        main_results['null_std_symmetric'] = null_stats['symmetric']['null_std']
        main_results['p_one_tailed_symmetric'] = null_stats['symmetric']['p_value_one_tailed']
        main_results['p_two_tailed_symmetric'] = null_stats['symmetric']['p_value_two_tailed']
    
    # Add per-feature contributions
    for i, feature_data in enumerate(results_dict['kl_per_feature']):
        main_results[f'{feature_names[i]}_KL'] = feature_data
    
    df = pd.DataFrame([main_results])
    
    return df


print("Master function with null distribution support defined.")

Master function with null distribution support defined.


## STEP 6: Video configuration and execution

**Purpose:** Run KL divergence analysis with optional null distribution testing - just set the session and parameters, then go. This expects that feature masks have been created for each worm in a given recorded video.

**Instructions:**
1. Set SESSION to your session prefix (date/time/trial)
2. Set `COMPUTE_NULL = True` to enable circshift null testing (adds ~30-60s per video)
3. Set `SHOW_NULL_DIAGNOSTIC = True/False` to show/hide KL vs shift plots
4. Run this cell
5. Results are saved automatically

**What it does:**
- Finds all Feature_vector.npy files matching your session
- Processes each one through KL divergence analysis
- **If COMPUTE_NULL=True**: Computes exhaustive circshift null distribution and p-values
- Saves individual JSON/CSV files per video
- Creates combined CSV with all results (including null statistics if enabled)
- Prints significance summary (how many worms show p < 0.05)

In [16]:
# ============================================================================
# CONFIGURATION SECTION
# ============================================================================

HOME = "/n/holylabs/LABS/gershman_lab/Users/zkelso/"
NETSCRATCH = "/n/netscratch/gershman_lab/Lab/zkelso/"
HOLYLABS = '/n/holylabs/LABS/gershman_lab/Users/zkelso/'

# Output directories
KL_RESULTS_DIR = os.path.join(HOLYLABS, 'KL_divergence_results')
INDIVIDUAL_METADATA_DIR = os.path.join(KL_RESULTS_DIR, 'individual_video_KL_metadata')
INDIVIDUAL_SESSION_DIR = os.path.join(KL_RESULTS_DIR, 'individual_session_KL_results')

# Create directories if they don't exist
os.makedirs(KL_RESULTS_DIR, exist_ok=True)
os.makedirs(INDIVIDUAL_METADATA_DIR, exist_ok=True)
os.makedirs(INDIVIDUAL_SESSION_DIR, exist_ok=True)

SAVE_RESULTS = True

# Null distribution settings
COMPUTE_NULL = False  # Set to False to skip circshift null computation (saves time)
SHOW_NULL_DIAGNOSTIC = False  # Set to False to suppress diagnostic plots

# ============================================================================
# END CONFIGURATION SECTION
# ============================================================================










# Normalize filters to lists for uniform processing
def normalize_filter(filter_value):
    """Convert filter to list format. None/empty → None, string → [string], list → list"""
    if filter_value is None or filter_value == '' or filter_value == []:
        return None
    if isinstance(filter_value, str):
        return [filter_value]
    return filter_value

VIDEO_PREFIX_FILTER = normalize_filter(VIDEO_PREFIX_FILTER)
VIDEO_GROUP_FILTER = normalize_filter(VIDEO_GROUP_FILTER)
EXCLUSION_FILTER = normalize_filter(EXCLUSION_FILTER)


# Display filter configuration
print("="*70)
print("KL DIVERGENCE BATCH PROCESSING")
print("="*70)
print(f"\nFilter Configuration:")
if VIDEO_PREFIX_FILTER:
    print(f"  Prefix filter: {VIDEO_PREFIX_FILTER}")
else:
    print(f"  Prefix filter: None (all prefixes)")

if VIDEO_GROUP_FILTER:
    print(f"  Group filter: {VIDEO_GROUP_FILTER}")
else:
    print(f"  Group filter: None (all groups)")

if EXCLUSION_FILTER:
    print(f"  Exclusion patterns: {EXCLUSION_FILTER}")
else:
    print(f"  Exclusion patterns: None (no exclusions)")


# Helper functions for filtering
def should_exclude(name, exclusion_patterns):
    """
    Check if name should be excluded based on substring matching.
    Returns (should_exclude: bool, matched_pattern: str or None)
    """
    if not exclusion_patterns:
        return False, None
    
    for pattern in exclusion_patterns:
        if pattern in name:
            return True, pattern
    return False, None


def matches_prefix(name, prefix_patterns):
    """Check if name starts with ANY of the prefix patterns"""
    if not prefix_patterns:  # None = match all
        return True
    return any(name.startswith(prefix) for prefix in prefix_patterns)


def matches_group(session_name, group_patterns):
    """Check if session name ends with _{ANY} of the group patterns (TC/TP)"""
    if not group_patterns:  # None = match all
        return True
    return any(session_name.endswith(f'_{group}') for group in group_patterns)


def extract_session_from_feature_file(filename,LABEL_TYPE):
    """
    Extract session name from feature file.
    Example: '2025_10_17_14_30_05_trial_1_TC_regions_100_200_fullvideo_Feature_vector.npy'
    Returns: '2025_10_17_14_30_05_trial_1_TC'
    """
    basename = os.path.basename(filename)
    # Remove '_Feature_vector.npy' suffix
    basename = basename.replace(f"_{LABEL_TYPE}_Feature_vector.npy", '')
    # Split on '_regions_' and take first part
    if '_regions_' in basename:
        return basename.split('_regions_')[0]
    return basename


# ============================================================================
# AUTO-DISCOVER AND FILTER SESSIONS
# ============================================================================

print("\n" + "="*70)
print("DISCOVERING FEATURE FILES")
print("="*70)

# Find all feature files
features_folder = os.path.join(NETSCRATCH, 'Features')
all_feature_files = glob.glob(os.path.join(features_folder, f"*_{LABEL_TYPE}_Feature_vector.npy"))

print(f"\nFound {len(all_feature_files)} total feature files in {features_folder}")

# Extract unique session names
all_sessions = []
for feature_file in all_feature_files:
    session = extract_session_from_feature_file(feature_file,LABEL_TYPE)
    all_sessions.append(session)

all_sessions = sorted([s for s in all_sessions if s.startswith('2025')])
print(f"Found {len(all_sessions)} unique sessions")

# Filter sessions by GROUP (TC/TP)
if VIDEO_GROUP_FILTER:
    group_filtered_sessions = [s for s in all_sessions if matches_group(s, VIDEO_GROUP_FILTER)]
    excluded_by_group = len(all_sessions) - len(group_filtered_sessions)
    if excluded_by_group > 0:
        group_list = ', '.join([f"'_{g}'" for g in VIDEO_GROUP_FILTER])
        print(f"FILTERING: Excluded {excluded_by_group} sessions not ending with {group_list}")
    all_sessions = group_filtered_sessions

# Filter sessions by exclusion patterns
excluded_sessions = []
filtered_sessions = []
for session in all_sessions:
    should_excl, matched = should_exclude(session, EXCLUSION_FILTER)
    if should_excl:
        excluded_sessions.append((session, matched))
    else:
        filtered_sessions.append(session)

if excluded_sessions:
    print(f"FILTERING: Excluded {len(excluded_sessions)} sessions matching exclusion patterns:")
    for session, pattern in excluded_sessions:
        print(f"  - {session} (matched '{pattern}')")

all_sessions = filtered_sessions
print(f"FILTERING: Proceeding with {len(all_sessions)} sessions after filtering")

# Now filter feature files within each session
SESSIONS_TO_PROCESS = []
excluded_videos = []

for session in all_sessions:
    pattern = os.path.join(features_folder, f"{session}_regions_*_Feature_vector.npy")
    session_feature_files = glob.glob(pattern)
    
    # Filter by prefix and exclusion
    filtered_files = []
    for feature_file in session_feature_files:
        basename = os.path.basename(feature_file)
        video_name = basename.replace(f"_{LABEL_TYPE}_Feature_vector.npy", '')
        
        # Check prefix
        if not matches_prefix(video_name, VIDEO_PREFIX_FILTER):
            continue
        
        # Check exclusion
        should_excl, matched = should_exclude(video_name, EXCLUSION_FILTER)
        if should_excl:
            excluded_videos.append((session, video_name, matched))
            continue
        
        filtered_files.append(feature_file)
    
    # Only add session if it has filtered files
    if filtered_files:
        SESSIONS_TO_PROCESS.append(session)

if excluded_videos:
    print(f"\nFILTERING: Excluded {len(excluded_videos)} videos matching exclusion patterns:")
    for session, video, pattern in excluded_videos:
        print(f"  - {video}")
        print(f"    (from session: {session}, matched '{pattern}')")

if not SESSIONS_TO_PROCESS:
    print(f"\nERROR: No sessions found matching filters")
    print(f"  Prefix: {VIDEO_PREFIX_FILTER}")
    print(f"  Group: {VIDEO_GROUP_FILTER}")
    print(f"  Exclusions: {EXCLUSION_FILTER}")
    print(f"  Searched in: {features_folder}")
    raise ValueError("No matching sessions found")

print(f"\n{'='*70}")
print(f"SESSIONS TO PROCESS: {len(SESSIONS_TO_PROCESS)}")
print(f"{'='*70}")
for idx, session in enumerate(SESSIONS_TO_PROCESS, 1):
    print(f"  {idx}. {session}")
print()


# No need to modify anything below this unless intentionally altering code.



# ============================================================================
# PROCESS EACH SESSION
# ============================================================================

all_sessions_results = []  # Store results from ALL sessions

for session_idx, SESSION in enumerate(SESSIONS_TO_PROCESS, 1):
    
    print("\n" + "="*70)
    print(f"SESSION {session_idx}/{len(SESSIONS_TO_PROCESS)}: {SESSION}")
    print("="*70 + "\n")
    
    # Find all feature files for current/given session
    pattern = os.path.join(NETSCRATCH, 'Features', f'{SESSION}_regions_*_Feature_vector.npy')
    feature_files = glob.glob(pattern)
    
    # Apply prefix and exclusion filtering to feature files
    filtered_feature_files = []
    for feature_file in feature_files:
        basename = os.path.basename(feature_file)
        video_name = basename.replace(f"_{LABEL_TYPE}_Feature_vector.npy", '')
        
        # Check prefix
        if not matches_prefix(video_name, VIDEO_PREFIX_FILTER):
            continue
        
        # Check exclusion
        should_excl, matched = should_exclude(video_name, EXCLUSION_FILTER)
        if should_excl:
            continue
        
        filtered_feature_files.append(feature_file)
    
    feature_files = filtered_feature_files
    
    if not feature_files:
        print(f"WARNING: No feature files found for {SESSION} after filtering")
        continue  # Skip to next session
    
    print(f"Found {len(feature_files)} videos")
    if COMPUTE_NULL:
        print(f"Null distribution: ENABLED")
    else:
        print(f"Null distribution: DISABLED")
    print()
    
    session_results = []  # Store results from current/given session
    
    # Process each video in this session
    for idx, feature_file in enumerate(feature_files, 1):
        if "FINAL" not in feature_file:
            continue
        else:
            video = os.path.basename(feature_file).replace(f'{LABEL_TYPE}_Feature_vector.npy', '')
            pdb.set_trace()
            regions = video.split('_regions_')[1].split('_fullvideo')[0]
            
            print(f"  [{idx}/{len(feature_files)}] regions_{regions}")
            
            try:
                df, results = analyze_video_kl_divergence(
                    video_name=video,
                    home_dir=HOME,
                    netscratch_dir=NETSCRATCH,
                    LABEL_TYPE= LABEL_TYPE,
                    save_results=False,  # We control saving here
                    compute_null=COMPUTE_NULL,
                    show_null_diagnostic=SHOW_NULL_DIAGNOSTIC
                )
                session_results.append(df)
                all_sessions_results.append(df)  # Also add to combined list
    
                kl = results['kl_results']['kl_all']
                print(f"      KL = {kl:.4f}")
                
                if COMPUTE_NULL and results['null_statistics'] is not None:
                    p_val = results['null_statistics']['on_given_off']['p_value_one_tailed']
                    print(f"      p-value (one-tailed) = {p_val:.4f}")
                
                # ================================================================
                # Save individual video metadata (JSON only - contains extra info)
                # ================================================================
                if SAVE_RESULTS:
                    individual_json = os.path.join(INDIVIDUAL_METADATA_DIR, f'{video}_KL_results.json')
                    
                    # Helper function to convert numpy types to native Python for JSON
                    def convert_numpy(obj):
                        if isinstance(obj, np.integer):
                            return int(obj)
                        elif isinstance(obj, np.floating):
                            return float(obj)
                        elif isinstance(obj, np.ndarray):
                            return obj.tolist()
                        elif isinstance(obj, dict):
                            return {k: convert_numpy(v) for k, v in obj.items()}
                        elif isinstance(obj, list):
                            return [convert_numpy(item) for item in obj]
                        else:
                            return obj
                    
                    # Convert results to JSON-serializable format
                    results_serializable = convert_numpy(results)
                    
                    with open(individual_json, 'w') as f:
                        json.dump(results_serializable, f, indent=2)
                    
                    print(f"      Metadata saved: {os.path.basename(individual_json)}")
                # ================================================================
                
            except Exception as e:
                print(f"      ERROR: {e}")
                traceback.print_exc()
                continue
        
    # Save per-session CSV in subdirectory
    if session_results and SAVE_RESULTS:
        session_df = pd.concat(session_results, ignore_index=True)
        
        print(f"\n  Session {SESSION} summary:")
        print(f"    Videos processed: {len(session_df)}")
        #print(f"    Mean KL: {['KL_CSon_given_CSoff'].mean():.4f}")
        
        if COMPUTE_NULL and 'p_one_tailed_CSon_given_CSoff' in session_df.columns:
            n_sig = np.sum(session_df['p_one_tailed_CSon_given_CSoff'] < 0.05)
            print(f"    Significant (p < 0.05): {n_sig}/{len(session_df)} worms")
        
        # Save per-session file in subdirectory
        session_csv = os.path.join(INDIVIDUAL_SESSION_DIR, f'{SESSION}_KL_results.csv')
        session_df.to_csv(session_csv, index=False)
        print(f"    Saved: {session_csv}")

            
            
# ============================================================================
# COMBINED RESULTS ACROSS ALL SESSIONS
# ============================================================================

if all_sessions_results:
    print("\n" + "="*70)
    print("COMBINED RESULTS - ALL SESSIONS")
    print("="*70)
    
    combined_df = pd.concat(all_sessions_results, ignore_index=True)
    
    print(f"Total videos: {len(combined_df)}")
    print(f"Total sessions: {len(SESSIONS_TO_PROCESS)}")
    #print(f"\nKL(CS-on || CS-off) across all sessions:")
    #print(f"  Mean:   {combined_df['KL_CSon_given_CSoff'].mean():.4f}")
    #print(f"  Median: {combined_df['KL_CSon_given_CSoff'].median():.4f}")
    #print(f"  Std:    {combined_df['KL_CSon_given_CSoff'].std():.4f}")
    #print(f"  Range:  {combined_df['KL_CSon_given_CSoff'].min():.4f} - {combined_df['KL_CSon_given_CSoff'].max():.4f}")
    
    if COMPUTE_NULL and 'p_one_tailed_CSon_given_CSoff' in combined_df.columns:
        print(f"\nNull hypothesis testing:")
        n_sig_05 = np.sum(combined_df['p_one_tailed_CSon_given_CSoff'] < 0.05)
        n_sig_01 = np.sum(combined_df['p_one_tailed_CSon_given_CSoff'] < 0.01)
        print(f"  Significant at p < 0.05: {n_sig_05}/{len(combined_df)} ({100*n_sig_05/len(combined_df):.1f}%)")
        print(f"  Significant at p < 0.01: {n_sig_01}/{len(combined_df)} ({100*n_sig_01/len(combined_df):.1f}%)")
    
    # Per-session breakdown
    print(f"\nPer-session summary:")
    #session_summary = combined_df.groupby('session')['KL_CSon_given_CSoff'].agg(['count', 'mean', 'std'])
    #print(session_summary)
    
    # Save master combined file with timestamp
    if SAVE_RESULTS:
        timestamp = datetime.datetime.now().strftime("%Y_%m_%d_%H_%M_%S")
        master_filename = f'Tasmanian_Conditioning_KL_Results_COMPILED_{timestamp}.csv'
        master_csv = os.path.join(KL_RESULTS_DIR, master_filename)
        combined_df.to_csv(master_csv, index=False)
        
        print(f"\n{'='*70}")
        print(f"MASTER FILE SAVED")
        print(f"{'='*70}")
        print(f"Filename: {master_filename}")
        print(f"Full path: {master_csv}")
        print(f"\nUse this file for learning curve plotting (KL vs. Day)")

else:
    print("\nNo results to combine - all sessions failed or had no videos.")

KL DIVERGENCE BATCH PROCESSING

Filter Configuration:
  Prefix filter: ['2025_10_14_10_25_19']
  Group filter: ['TC']
  Exclusion patterns: None (no exclusions)

DISCOVERING FEATURE FILES

Found 6 total feature files in /n/netscratch/gershman_lab/Lab/zkelso/Features
Found 6 unique sessions
FILTERING: Proceeding with 6 sessions after filtering

SESSIONS TO PROCESS: 6
  1. 2025_10_14_10_25_19_trial_1_TC
  2. 2025_10_14_10_25_19_trial_1_TC
  3. 2025_10_14_10_25_19_trial_1_TC
  4. 2025_10_14_10_25_19_trial_1_TC
  5. 2025_10_14_10_25_19_trial_1_TC
  6. 2025_10_14_10_25_19_trial_1_TC


SESSION 1/6: 2025_10_14_10_25_19_trial_1_TC

Found 6 videos
Null distribution: DISABLED

> /tmp/ipykernel_3972871/1419032064.py(278)<module>()
    276             video = os.path.basename(feature_file).replace(f'{LABEL_TYPE}_Feature_vector.npy', '')
    277             pdb.set_trace()
--> 278             regions = video.split('_regions_')[1].split('_fullvideo')[0]
    279 
    280             print(f"  [{idx}/

ipdb>  c


  [6/6] regions_1072_463_1341_1389
KL DIVERGENCE ANALYSIS
Video: 2025_10_14_10_25_19_trial_1_TC_regions_1072_463_1341_1389_fullvideo_frame_split_FINAL_
Session: 2025_10_14_10_25_19_trial_1_TC

Loading data...
  Feature vector: (12, 1799)
  Trial definitions: 3 trials
Extracting CS-on frames (during trials)...
  Trial 1: frames 18-292 (275 frames)
  Trial 2: frames 618-892 (275 frames)
  Trial 3: frames 1218-1492 (275 frames)
Total CS-on frames: 825

Extracting CS-off frames (inter-trial intervals only)...
  After trial 1: frames 293-617 (325 frames)
  After trial 2: frames 893-1217 (325 frames)
  After trial 3: frames 1493-1799 (307 frames)
Total CS-off frames: 956

Transposing to (samples, features) format...
  CS-on:  (825, 12)
  CS-off: (956, 12)

NaN analysis:
Feature              CS-on NaNs      CS-off NaNs     CS-on %    CS-off %  
----------------------------------------------------------------------
Areas                0               0               0.00       0.00      
Area

ipdb>  c


  [6/6] regions_1072_463_1341_1389
KL DIVERGENCE ANALYSIS
Video: 2025_10_14_10_25_19_trial_1_TC_regions_1072_463_1341_1389_fullvideo_frame_split_FINAL_
Session: 2025_10_14_10_25_19_trial_1_TC

Loading data...
  Feature vector: (12, 1799)
  Trial definitions: 3 trials
Extracting CS-on frames (during trials)...
  Trial 1: frames 18-292 (275 frames)
  Trial 2: frames 618-892 (275 frames)
  Trial 3: frames 1218-1492 (275 frames)
Total CS-on frames: 825

Extracting CS-off frames (inter-trial intervals only)...
  After trial 1: frames 293-617 (325 frames)
  After trial 2: frames 893-1217 (325 frames)
  After trial 3: frames 1493-1799 (307 frames)
Total CS-off frames: 956

Transposing to (samples, features) format...
  CS-on:  (825, 12)
  CS-off: (956, 12)

NaN analysis:
Feature              CS-on NaNs      CS-off NaNs     CS-on %    CS-off %  
----------------------------------------------------------------------
Areas                0               0               0.00       0.00      
Area

...

  [6/6] regions_1072_463_1341_1389
KL DIVERGENCE ANALYSIS
Video: 2025_10_14_10_25_19_trial_1_TC_regions_1072_463_1341_1389_fullvideo_frame_split_FINAL_
Session: 2025_10_14_10_25_19_trial_1_TC

Loading data...
  Feature vector: (12, 1799)
  Trial definitions: 3 trials
Extracting CS-on frames (during trials)...
  Trial 1: frames 18-292 (275 frames)
  Trial 2: frames 618-892 (275 frames)
  Trial 3: frames 1218-1492 (275 frames)
Total CS-on frames: 825

Extracting CS-off frames (inter-trial intervals only)...
  After trial 1: frames 293-617 (325 frames)
  After trial 2: frames 893-1217 (325 frames)
  After trial 3: frames 1493-1799 (307 frames)
Total CS-off frames: 956

Transposing to (samples, features) format...
  CS-on:  (825, 12)
  CS-off: (956, 12)

NaN analysis:
Feature              CS-on NaNs      CS-off NaNs     CS-on %    CS-off %  
----------------------------------------------------------------------
Areas                0               0               0.00       0.00      
Area

ipdb>  c


  [6/6] regions_1072_463_1341_1389
KL DIVERGENCE ANALYSIS
Video: 2025_10_14_10_25_19_trial_1_TC_regions_1072_463_1341_1389_fullvideo_frame_split_FINAL_
Session: 2025_10_14_10_25_19_trial_1_TC

Loading data...
  Feature vector: (12, 1799)
  Trial definitions: 3 trials
Extracting CS-on frames (during trials)...
  Trial 1: frames 18-292 (275 frames)
  Trial 2: frames 618-892 (275 frames)
  Trial 3: frames 1218-1492 (275 frames)
Total CS-on frames: 825

Extracting CS-off frames (inter-trial intervals only)...
  After trial 1: frames 293-617 (325 frames)
  After trial 2: frames 893-1217 (325 frames)
  After trial 3: frames 1493-1799 (307 frames)
Total CS-off frames: 956

Transposing to (samples, features) format...
  CS-on:  (825, 12)
  CS-off: (956, 12)

NaN analysis:
Feature              CS-on NaNs      CS-off NaNs     CS-on %    CS-off %  
----------------------------------------------------------------------
Areas                0               0               0.00       0.00      
Area

ipdb>  c


  [6/6] regions_1072_463_1341_1389
KL DIVERGENCE ANALYSIS
Video: 2025_10_14_10_25_19_trial_1_TC_regions_1072_463_1341_1389_fullvideo_frame_split_FINAL_
Session: 2025_10_14_10_25_19_trial_1_TC

Loading data...
  Feature vector: (12, 1799)
  Trial definitions: 3 trials
Extracting CS-on frames (during trials)...
  Trial 1: frames 18-292 (275 frames)
  Trial 2: frames 618-892 (275 frames)
  Trial 3: frames 1218-1492 (275 frames)
Total CS-on frames: 825

Extracting CS-off frames (inter-trial intervals only)...
  After trial 1: frames 293-617 (325 frames)
  After trial 2: frames 893-1217 (325 frames)
  After trial 3: frames 1493-1799 (307 frames)
Total CS-off frames: 956

Transposing to (samples, features) format...
  CS-on:  (825, 12)
  CS-off: (956, 12)

NaN analysis:
Feature              CS-on NaNs      CS-off NaNs     CS-on %    CS-off %  
----------------------------------------------------------------------
Areas                0               0               0.00       0.00      
Area